In [1]:
# Heatmap showing the Mean Absolute Error (MAE) for each deconvolution method and samples with gene removed across the better performing matrix 

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'TOO-Decon-Degradation/merged_Results_Spillover_Random_TOOv2.txt'
df = pd.read_csv(file_path, sep='\t')

# === EXTRACT NUMERIC GENE REMOVAL VALUE FOR ORDERING ===
df['GeneRemovalNumeric'] = (
    df['GeneRemoval']
    .str.extract(r'top_(\d+)')
    .astype(float)
)

# === PIVOT TO METHOD × GENE REMOVAL ===
pivot_df = df.pivot_table(
    index='Method',
    columns='GeneRemovalNumeric',
    values='Average',     # MAE values
    aggfunc='mean'
).sort_index(axis=1)

# === REORDER ROWS ===
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
pivot_df = pivot_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(6, 4), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    pivot_df,
    cmap='viridis_r',             
#    annot=True,
#    fmt=".2f",
#    annot_kws={"size": 7},
    cbar_kws={'shrink': 0.9},
    linewidths=0,
    linecolor='white'
)

# Force ticks to show
ax.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6
)
ax.tick_params(
    axis='y', which='both', left=True, right=False, labelleft=True,
    length=4, width=0.6
)

# Colorbar styling
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Mean Absolute Error", fontsize=13, rotation=270, labelpad=20)   
cbar.ax.tick_params(labelsize=11, width=0.8, length=4)             
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}"))

# === Axes styling ===
plt.ylabel('Deconvolution Tool', fontsize=14, labelpad=10)
plt.xlabel('Genes Removed (%)', fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# === SAVE FIGURE ===
fig.savefig("MAE_Heatmap_Best-Matrix_Degradation_Clean.svg")
plt.close(fig)
